In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain.agents import create_agent
from langchain_core.prompts import PromptTemplate

from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

import json

In [ ]:
llm = ChatOllama(
    model="gpt-oss:20b",
    temperature=0.7,
    num_predict=1000,
)

In [ ]:
messages = []

def add_system_message(messages, message: str):
    messages.append(SystemMessage(content=message))
    return

def add_user_message(messages,message: str):
    messages.append(HumanMessage(content=message))
    return

def add_assistant_message(messages, message: str):
    messages.append(AIMessage(content=message))
    return

In [ ]:
def chat(messages):
    response = llm.invoke(messages)
    return response

In [ ]:
def generate_dataset():
    messages = []
    
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    response = llm.bind(temperature=0.8).invoke(messages, stop=["```"])

    data = json.loads(response.content)

    with open("demo-resources/dataset.json", "w", encoding="utf-8") as f:
        json.dump(data, f,indent=4, ensure_ascii=False)

    return response


In [ ]:
# dataset = generate_dataset()

In [ ]:
def run_prompt(test_case):
    prompt = f"""
        Please solve the following task:
        {test_case['task']}
* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    response = llm.bind(options={"temperature": 0.2}).invoke(messages)
    
    # print(f"SOLUTION:\n{response.content}")
    
    return response.content


In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    # response = llm.invoke(messages, stop=["```"])
    response = llm.bind(options={"temperature": 0.0, "stop": ["```"]}).invoke(messages)

    # print(f"Evaluation:\n{response.content}")

    return json.loads(response.content)

In [ ]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    print(f"Syntax score: {syntax_score}")

    score = (model_score + syntax_score) / 2

    return {
        "response": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:

from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])

    print(f"Average Score: {average_score:.2f}")
    return results

In [ ]:
with open("demo-resources/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=4, ensure_ascii=False))

Syntax score: 10
Syntax score: 10
Syntax score: 10
Average Score: 8.83
[
    {
        "response": "{\n  \"Resources\": {\n    \"MyBucket\": {\n      \"Type\": \"AWS::S3::Bucket\",\n      \"Properties\": {\n        \"BucketName\": { \"Fn::Sub\": \"my-unique-bucket-${AWS::AccountId}\" },\n        \"VersioningConfiguration\": {\n          \"Status\": \"Enabled\"\n        },\n        \"PublicAccessBlockConfiguration\": {\n          \"BlockPublicAcls\": true,\n          \"IgnorePublicAcls\": true,\n          \"BlockPublicPolicy\": true,\n          \"RestrictPublicBuckets\": true\n        }\n      }\n    }\n  }\n}",
        "test_case": {
            "task": "Generate a CloudFormation template snippet in JSON that creates an S3 bucket named `my-unique-bucket-{account-id}` with versioning enabled and public read access blocked.",
            "format": "json",
            "solution_criteria": [
                "The output must be valid JSON.",
                "It should contain a single resou